# Publicador sintético — instrutor (Firebase Admin SDK)

Use esta versão somente no ambiente do instrutor. A escrita é feita com **Firebase Admin SDK**, portanto os alunos podem permanecer com regras de somente leitura.

No Google Colab, crie um Secret chamado `FIREBASE_SERVICE_ACCOUNT_JSON` contendo o JSON da conta de serviço. **Nunca compartilhe esse segredo.**


In [ ]:
!pip -q install firebase-admin


## 1. Configuração
Copie a **URL do Realtime Database** mostrada no console do Firebase. Não acrescente `/oficina/live.json`.

In [ ]:
# URL do Realtime Database.
FIREBASE_URL = "Coloque aqui o endereço "

# Modo de geração dos sinais. Exemplos: NORMAL, NOISE, BASELINE, MOTION, SPIKE, CLIPPING, OUTLIER, DROPOUT ou AUTO.
MODO = "NORMAL"

# Tempo total, em segundos, durante o qual o publicador ficará enviando pacotes ao Firebase.
DURACAO_SEGUNDOS = 100


In [ ]:
# Biblioteca padrão para controlar intervalos de tempo e obter timestamps.
import time

# Biblioteca padrão para gerar um identificador único para cada sessão de publicação.
import uuid

# Biblioteca padrão para converter o texto JSON armazenado no Secret do Colab em um dicionário Python.
import json

# NumPy é usado para gerar e manipular numericamente os sinais sintéticos de ECG e PPG.
import numpy as np

# SDK administrativo do Firebase; permite ao publicador escrever no banco usando uma conta de serviço.
import firebase_admin

# credentials lê a conta de serviço; db dá acesso ao Realtime Database.
from firebase_admin import credentials, db

# userdata permite ler Secrets armazenados de forma protegida no Google Colab.
from google.colab import userdata


# Frequência de amostragem dos sinais sintéticos, em amostras por segundo (Hz).
FS = 250

# Quantidade de amostras de cada sinal enviada ao Firebase em cada pacote.
AMOSTRAS_POR_PACOTE = 50

# Frequência cardíaca usada como referência na geração dos sinais sintéticos.
HR_BPM = 72

# Gerador de números aleatórios com semente fixa.
# A semente fixa torna os ruídos/artefatos reproduzíveis entre execuções.
rng = np.random.default_rng(12345)


# Lista dos modos aceitos quando MODO não é "AUTO".
MODOS_VALIDOS = [
    "NORMAL",
    "NOISE",
    "BASELINE",
    "MOTION",
    "SPIKE",
    "CLIPPING",
    "OUTLIER",
    "DROPOUT",
]


# Gera uma curva gaussiana circular.
# Ela é usada como bloco matemático para construir as ondas P, Q, R, S e T do ECG sintético.
def _gauss_circular(fase, mu, sigma):
    # Calcula a menor distância entre a fase atual e o centro da gaussiana,
    # considerando que a fase cardíaca se repete ciclicamente entre 0 e 1.
    d = np.minimum(np.abs(fase - mu), 1.0 - np.abs(fase - mu))

    # Retorna a gaussiana; sigma controla a largura da onda gerada.
    return np.exp(-0.5 * (d / sigma) ** 2)


# Gera um bloco de ECG e dois canais de PPG sintéticos.
# sample_start indica o índice da primeira amostra do bloco atual;
# n é a quantidade de amostras a gerar.
def gerar_sinais(sample_start, n, fs=FS, hr_bpm=HR_BPM):
    """Gera ECG e PPG sintéticos para a atividade didática."""

    # Cria os índices absolutos das amostras deste pacote.
    # Usar índices absolutos evita reiniciar o sinal a cada novo pacote.
    idx = np.arange(sample_start, sample_start + n)

    # Converte os índices das amostras em tempo, em segundos.
    t = idx / fs

    # Período de um batimento cardíaco em segundos.
    periodo = 60.0 / hr_bpm

    # Converte o tempo em uma fase normalizada entre 0 e 1 dentro de cada batimento.
    fase = (t % periodo) / periodo

    # ECG sintético.
    # Cada termo gaussiano aproxima uma onda do complexo P-QRS-T.
    ecg = (
        0.08 * _gauss_circular(fase, 0.18, 0.025)      # Onda P
        - 0.12 * _gauss_circular(fase, 0.365, 0.012)  # Onda Q
        + 1.00 * _gauss_circular(fase, 0.400, 0.010)  # Onda R
        - 0.25 * _gauss_circular(fase, 0.435, 0.014)  # Onda S
        + 0.30 * _gauss_circular(fase, 0.68, 0.055)   # Onda T
        + 0.015 * np.sin(2 * np.pi * 0.20 * t)        # Pequena variação lenta da linha de base
    )

    # Introduz no PPG um atraso aproximado de 180 ms em relação ao ECG.
    # Isso imita o tempo entre a atividade elétrica cardíaca e a chegada da onda de pulso.
    fase_ppg = ((t - 0.18) % periodo) / periodo

    # Gera a componente principal do pulso.
    # clip() limita o argumento e maximum() impede valores negativos.
    pulse = np.maximum(
        np.sin(np.pi * np.clip(fase_ppg / 0.55, 0, 1)),
        0,
    ) ** 2

    # Mantém o pulso somente na parte desejada do ciclo.
    pulse *= (fase_ppg < 0.55)

    # Pequena onda dicrótica adicionada ao PPG.
    dicrotic = 0.15 * _gauss_circular(fase_ppg, 0.62, 0.035)

    # Soma as componentes que formam o pulso fotopletismográfico.
    ppg_base = pulse + dicrotic

    # Canal infravermelho do PPG.
    # O valor DC elevado imita o nível de leitura bruto típico de um sensor óptico.
    ppg_ir = (
        50000
        + 6500 * ppg_base
        + 120 * np.sin(2 * np.pi * 0.08 * t)
    )

    # Canal vermelho do PPG, com amplitude e fase ligeiramente diferentes.
    ppg_red = (
        42000
        + 4300 * ppg_base
        + 100 * np.sin(2 * np.pi * 0.08 * t + 0.2)
    )

    # Converte explicitamente os sinais para float antes de retorná-los.
    return (
        t,
        ecg.astype(float),
        ppg_ir.astype(float),
        ppg_red.astype(float),
    )


# Aplica ao pacote atual um artefato específico para fins didáticos.
# O modo NORMAL retorna os sinais sem alteração.
def aplicar_falha(modo, t, ecg, ppg_ir, ppg_red):
    # Padroniza o texto para maiúsculas, evitando diferenças como "noise" e "NOISE".
    modo = modo.upper()

    if modo == "NORMAL":
        return ecg, ppg_ir, ppg_red

    if modo == "NOISE":
        # Acrescenta ruído aleatório aos três sinais.
        ecg = ecg + rng.normal(0, 0.16, len(ecg))
        ppg_ir = ppg_ir + rng.normal(0, 700, len(ppg_ir))
        ppg_red = ppg_red + rng.normal(0, 550, len(ppg_red))

    elif modo == "BASELINE":
        # Simula deriva lenta da linha de base do ECG.
        ecg = ecg + 0.42 * np.sin(2 * np.pi * 0.27 * t)

    elif modo == "MOTION":
        # Combina uma oscilação periódica e ruído para simular artefato de movimento.
        artefato = (
            0.40 * np.sin(2 * np.pi * 1.3 * t)
            + rng.normal(0, 0.10, len(ecg))
        )

        # Aplica o movimento ao ECG e aos dois canais ópticos.
        ecg = ecg + artefato
        ppg_ir = ppg_ir + 2500 * np.sin(2 * np.pi * 1.3 * t)
        ppg_red = ppg_red + 1800 * np.sin(2 * np.pi * 1.3 * t)

    elif modo == "SPIKE":
        # Seleciona aleatoriamente cerca de 4% das amostras para receber impulsos.
        mask = rng.random(len(ecg)) < 0.04

        # copy() evita alterar o vetor original recebido pela função.
        ecg = ecg.copy()

        # Acrescenta impulsos positivos ou negativos nas posições selecionadas.
        ecg[mask] += rng.choice([-1.8, 1.8], size=mask.sum())

    elif modo == "CLIPPING":
        # Limita artificialmente a amplitude, simulando saturação do sistema de aquisição.
        ecg = np.clip(ecg, -0.18, 0.55)
        ppg_ir = np.clip(ppg_ir, 49000, 54500)

    elif modo == "OUTLIER":
        # Só executa se o pacote possuir pelo menos uma amostra.
        if len(ecg):
            # Escolhe aleatoriamente uma posição do pacote.
            k = int(rng.integers(0, len(ecg)))

            # Cria cópias antes de alterar pontualmente os vetores.
            ecg = ecg.copy()
            ppg_ir = ppg_ir.copy()

            # Insere valores deliberadamente anormais.
            ecg[k] = 4.5
            ppg_ir[k] = 90000

    # DROPOUT não modifica as amostras aqui.
    # A perda de pacote é simulada depois, na função publicar().
    return ecg, ppg_ir, ppg_red


# Inicializa o Firebase Admin SDK e devolve uma referência ao nó /oficina/live.
# O JSON da conta de serviço é lido do Secret FIREBASE_SERVICE_ACCOUNT_JSON do Colab.
def iniciar_firebase_admin(firebase_url=FIREBASE_URL):
    # Evita inicializar o Firebase uma segunda vez se esta célula for executada novamente.
    if not firebase_admin._apps:

        # Recupera o JSON armazenado no Secret e converte o texto para um dicionário Python.
        cred_info = json.loads(
            userdata.get("FIREBASE_SERVICE_ACCOUNT_JSON")
        )

        # Cria uma credencial administrativa a partir da conta de serviço.
        cred = credentials.Certificate(cred_info)

        # Inicializa a conexão do Admin SDK com o Realtime Database indicado por firebase_url.
        firebase_admin.initialize_app(
            cred,
            {"databaseURL": firebase_url},
        )

    # Retorna uma referência direta ao nó usado pela oficina.
    # O Admin SDK escreverá neste caminho, independentemente das regras públicas de escrita.
    return db.reference("oficina/live")


# Gera os sinais continuamente e publica um novo pacote no Firebase a cada intervalo de amostragem.
# modo define o artefato aplicado; duracao_s define por quanto tempo o laço permanecerá ativo.
def publicar(modo="NORMAL", duracao_s=100):
    # Padroniza a escrita do modo.
    modo = modo.upper()

    # Verifica se o modo solicitado existe.
    if modo != "AUTO" and modo not in MODOS_VALIDOS:
        raise ValueError(f"Modo inválido: {modo}")

    # Inicializa o Firebase e obtém a referência de /oficina/live.
    ref = iniciar_firebase_admin()

    # Gera um identificador único para esta execução do publicador.
    # Os clientes podem usar esse valor para perceber que uma nova sessão começou.
    session_id = uuid.uuid4().hex

    print("Destino: /oficina/live (Admin SDK)")
    print("Sessão:", session_id)

    # Número sequencial do pacote publicado.
    seq = 0

    # Índice absoluto da primeira amostra do próximo pacote.
    sample_index = 0

    # Marca o instante em que esta sessão começou.
    inicio = time.time()

    # Guarda o instante planejado para a próxima publicação.
    # Esse controle reduz a deriva temporal causada pelo tempo de execução do código.
    proximo = inicio

    # Contadores apenas informativos.
    enviados = 0
    omitidos = 0

    try:
        # Mantém o publicador ativo até atingir duracao_s.
        while time.time() - inicio < duracao_s:
            # Tempo transcorrido desde o início da publicação.
            elapsed = time.time() - inicio

            if modo == "AUTO":
                # No modo AUTO, troca de cenário aproximadamente a cada 12 segundos.
                modos_auto = [
                    "NORMAL",
                    "NOISE",
                    "BASELINE",
                    "MOTION",
                    "SPIKE",
                    "CLIPPING",
                    "OUTLIER",
                    "DROPOUT",
                ]

                # // é divisão inteira; % faz a lista reiniciar após o último modo.
                atual = modos_auto[
                    int(elapsed // 12) % len(modos_auto)
                ]
            else:
                # Nos demais casos, usa sempre o modo selecionado pelo instrutor.
                atual = modo

            # Gera o próximo bloco contínuo de amostras.
            t, ecg, ppg_ir, ppg_red = gerar_sinais(
                sample_index,
                AMOSTRAS_POR_PACOTE,
            )

            # Aplica o artefato correspondente ao modo atual.
            ecg, ppg_ir, ppg_red = aplicar_falha(
                atual,
                t,
                ecg,
                ppg_ir,
                ppg_red,
            )

            # No modo DROPOUT, aproximadamente 25% dos pacotes deixam de ser enviados.
            enviar = not (
                atual == "DROPOUT"
                and rng.random() < 0.25
            )

            if enviar:
                # Monta o objeto que será gravado como JSON no Realtime Database.
                pacote = {
                    "session_id": session_id,       # Identifica a sessão atual.
                    "seq": seq,                     # Número sequencial deste pacote.
                    "sample_start": sample_index,   # Índice absoluto da primeira amostra.
                    "timestamp_pub": time.time(),   # Horário Unix da publicação.
                    "fs": FS,                       # Frequência de amostragem em Hz.
                    "n": AMOSTRAS_POR_PACOTE,       # Número de amostras em cada vetor.
                    "source": "SIM",                # Informa que a fonte é sintética.
                    "ecg": np.round(ecg, 5).tolist(),        # ECG convertido para lista JSON.
                    "ppg_ir": np.round(ppg_ir, 2).tolist(),  # PPG infravermelho.
                    "ppg_red": np.round(ppg_red, 2).tolist() # PPG vermelho.
                }

                # Substitui o conteúdo atual de /oficina/live pelo novo pacote.
                # set() é apropriado aqui porque queremos manter apenas o pacote mais recente.
                ref.set(pacote)

                enviados += 1
            else:
                # Conta os pacotes deliberadamente omitidos no modo DROPOUT.
                omitidos += 1

            # Imprime o estado a cada cinco pacotes para não poluir demais a saída do Colab.
            if seq % 5 == 0:
                print(
                    f"\rtempo={elapsed:6.1f}s  "
                    f"modo={atual:9s}  "
                    f"seq={seq:5d}  "
                    f"sample={sample_index:7d}  "
                    f"enviados={enviados:4d}  "
                    f"omitidos={omitidos:3d}",
                    end="",
                    flush=True,
                )

            # Prepara os índices do próximo pacote.
            seq += 1
            sample_index += AMOSTRAS_POR_PACOTE

            # Cada pacote representa AMOSTRAS_POR_PACOTE / FS segundos.
            # Com 50 amostras a 250 Hz, o intervalo esperado é de 0,2 s.
            proximo += AMOSTRAS_POR_PACOTE / FS

            # Aguarda somente o tempo restante até o próximo instante programado.
            # max(0, ...) evita passar tempo negativo para time.sleep().
            time.sleep(
                max(0, proximo - time.time())
            )

    except KeyboardInterrupt:
        # Permite encerrar o laço manualmente pelo botão "Interromper execução" do Colab.
        print("\nInterrompido pelo usuário.")

    finally:
        # Esta mensagem aparece tanto no término normal quanto após uma interrupção.
        print("\nPublicação encerrada.")


## 2. Iniciar publicação
Primeiro deixe os notebooks dos participantes aguardando. Depois execute esta célula.

In [ ]:
# Inicia o publicador usando o modo e a duração definidos na seção de configuração.
# A execução permanece nesta célula até atingir DURACAO_SEGUNDOS ou ser interrompida manualmente.
publicar(modo=MODO, duracao_s=DURACAO_SEGUNDOS)
